# Импорты

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
import os
import random
from torch.cuda.amp import autocast, GradScaler
warnings.filterwarnings('ignore')

In [2]:
def set_seed(seed=42):
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Seed set to {seed} (deterministic mode)")

# Метрика

In [3]:
def wape_plus_rbias(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    s = y_true.sum()
    if s == 0:
        return 0.0
    return np.abs(y_pred - y_true).sum() / s + np.abs(y_pred.sum() / s - 1)

# Загрузка данных

In [4]:
train = pd.read_parquet('train_team_track.parquet')
test = pd.read_parquet('test_team_track.parquet')
train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp'] = pd.to_datetime(test['timestamp'])

route_to_office = train.groupby('route_id')['office_from_id'].first().to_dict()
test['office_from_id'] = test['route_id'].map(route_to_office)

status_cols = [f'status_{i}' for i in range(1, 9)]
status_cols =  [col for col in status_cols if col not in ['status_1', 'status_3']]
for c in status_cols:
    test[c] = np.nan

WINDOW_SIZE = 48
FORECAST_HORIZON = 10

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
set_seed()
        
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Seed set to 42 (deterministic mode)
Device: cuda


# Конфигурации

In [5]:
MODE = 'final'

configs = {
    'experiment': {
        'sample_routes': 200,
        'epochs': 12,
        'patience': 4,
        'batch_size': 2048
    },
    'validation': {
        'sample_routes': 500,
        'epochs': 20,
        'patience': 5,
        'batch_size': 2048
    },
    'final': {
        'sample_routes': None,
        'epochs': 70,
        'patience': 10,
        'batch_size': 2048
    }
}

cfg = configs[MODE]
print(f"MODE: {MODE}")
print(f"Config: {cfg}")

EPOCHS = cfg['epochs']
PATIENCE = cfg['patience']
BS = cfg['batch_size']

MODE: final
Config: {'sample_routes': None, 'epochs': 70, 'patience': 10, 'batch_size': 2048}


### Смесь

In [ ]:
if cfg['sample_routes']:
    sample_routes = np.random.choice(
        train['route_id'].unique(),
        size=cfg['sample_routes'],
        replace=False
    )
#     wrost_routes = [268, 675, 897, 733, 717, 519]
#     sample_routes[sample_routes.shape[0] - len(wrost_routes):] = wrost_routes
    train = train[train['route_id'].isin(sample_routes)].copy()
train = train.reset_index(drop=True)
# np.save('sample_routes_experiment.npy', sample_routes)

### Плохие без невозможных

In [ ]:
bad_routes = [
    243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
    31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
    316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
    539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
    324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
    775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
    125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
    350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
    728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
    50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
    58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
    675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
    563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
    141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
    162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
    733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
    848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
    262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
    286, 137, 46, 340
]
very_bad_routes = [243, 124, 370, 215, 407, 977, 928, 364, 931, 505, 21, 531, 474, 450, 968, 582, 304, 518]
# other_bad = [42,87,126,136,328,377,413,431,437,454,560,569,577,609,651,678,775,811,846,874,875,902,912,916, 927, 304, 518]

available_routes = np.setdiff1d(bad_routes, very_bad_routes)
# available_routes = np.setdiff1d(available_routes, other_bad)

# sample_routes = np.random.choice(
#     available_routes,
#     size=cfg['sample_routes'],
#     replace=False
# )
train = train[train['route_id'].isin(available_routes)].copy()
train = train.reset_index(drop=True)

In [6]:
rids_target_zero = train.groupby('route_id')['target_2h'].mean()[lambda x: x <= 0.1].index.tolist()
print(len(rids_target_zero))
train = train[train.route_id.isin(np.setdiff1d(train.route_id.unique(), rids_target_zero))]
rids_s1_s3_zero = train.groupby('route_id')[['status_1', 'status_3']].mean().query('status_1 <= 0.1 and status_3 <= 0.1').index.tolist()
train = train[train.route_id.isin(rids_s1_s3_zero)]
train.reset_index(drop=True)
print(len(train.route_id.unique()))

16
310


### Без плохих

In [ ]:
rids_target_zero = train.groupby('route_id')['target_2h'].mean()[lambda x: x <= 0.1].index.tolist()
print(len(rids_target_zero))
train = train[train.route_id.isin(np.setdiff1d(train.route_id.unique(), rids_target_zero))]
rids_s1_s3_zero = train.groupby('route_id')[['status_1', 'status_3']].mean().query('status_1 <= 0.1 and status_3 <= 0.1').index.tolist()
print(len(rids_s1_s3_zero))
train = train[train.route_id.isin(np.setdiff1d(train.route_id.unique(), rids_s1_s3_zero))]
train.reset_index(drop=True)
print(len(train.route_id.unique()))

In [ ]:
if cfg['sample_routes']:
    exclude_routes = [
        243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
        31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
        316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
        539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
        324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
        775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
        125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
        350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
        728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
        50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
        58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
        675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
        563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
        141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
        162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
        733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
        848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
        262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
        286, 137, 46, 340
    ]

    available_routes = np.setdiff1d(train['route_id'].unique(), exclude_routes)
    
    sample_routes = np.random.choice(
        available_routes,
        size=min(cfg['sample_routes'], len(available_routes)),
        replace=False
    )
    
    train = train[train['route_id'].isin(sample_routes)].copy()
else:
    exclude_routes = [
        243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
        31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
        316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
        539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
        324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
        775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
        125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
        350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
        728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
        50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
        58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
        675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
        563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
        141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
        162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
        733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
        848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
        262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
        286, 137, 46, 340
    ]
#     bad_routes_dense = [445, 511, 261, 813, 432, 237, 24, 479, 607, 621]
    bad_3 = [7, 41, 117, 132, 164, 199, 218, 222, 259, 299, 313, 325, 347, 368, 380, 
             415, 438, 479, 521, 535, 648, 701, 754, 757, 760, 816, 840, 856, 861, 866, 
             876, 904, 940, 955, 971, 973, 978, 985, 991, 994]
    bad_4 = [180, 271, 482, 989]
    bad_5 = [180, 482]
    bad_6 = [180, 482]
    bad_7 = []
    bad_8 = [710]
    
    available_routes = np.setdiff1d(train['route_id'].unique(), exclude_routes)
    available_routes = np.setdiff1d(available_routes, bad_3)
    available_routes = np.setdiff1d(available_routes, bad_4)
    available_routes = np.setdiff1d(available_routes, bad_5)
    available_routes = np.setdiff1d(available_routes, bad_6)
    available_routes = np.setdiff1d(available_routes, bad_8)
    train = train[train['route_id'].isin(available_routes)].copy()
    
train = train.reset_index(drop=True)

### Загрузка созданного датасета

In [ ]:
if cfg['sample_routes']:
    sample_routes = np.load('sample_routes_experiment.npy')    
    train = train[train['route_id'].isin(sample_routes)].copy()
train = train.reset_index(drop=True)

# Скейлеры

In [7]:
class LogRobustScaler:
    def __init__(self):
        self.medians = {}
        self.iqrs = {}
        self.use_log = {}

    def fit(self, df, columns, log_columns=None):
        log_columns = log_columns or []
        for col in columns:
            v = df[col].dropna().values.copy()
            if col in log_columns:
                v = np.log1p(np.clip(v, 0, None))
                self.use_log[col] = True
            else:
                self.use_log[col] = False
            q25, q75 = np.percentile(v, [25, 75])
            self.medians[col] = np.median(v)
            self.iqrs[col] = max(q75 - q25, 1e-4)
        return self

    def transform(self, values, col):
        v = np.array(values, dtype=float)
        if self.use_log.get(col, False):
            v = np.log1p(np.clip(v, 0, None))
        return (v - self.medians[col]) / self.iqrs[col]

    def inverse_transform(self, values, col):
        v = np.array(values, dtype=float)
        v = v * self.iqrs[col] + self.medians[col]
        if self.use_log.get(col, False):
            v = np.expm1(v)
        return v

In [8]:
log_cols = status_cols
scaler = LogRobustScaler()
scaler.fit(train, log_cols, log_columns=log_cols)

target_scaler = LogRobustScaler()
target_scaler.fit(train, ['target_2h'])

# Создание фичей

In [9]:
def add_time_features(df):
    df = df.copy()
    df['time_slot'] = df['timestamp'].dt.hour * 2 + df['timestamp'].dt.minute // 30
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute/60
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 48)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 48)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    return df

train = add_time_features(train)
test = add_time_features(test)

In [10]:
time_features = ['hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
                 'dow_sin', 'dow_cos', 'is_weekend']

seq_cols = status_cols + ['target_2h']
num_seq_features = len(seq_cols) + len(time_features)

# Построение окон

### Дефолтное

In [12]:
print("Building sequences efficiently...")

train_sorted = train

for c in status_cols:
    train_sorted[f'{c}_norm'] = scaler.transform(train_sorted[c].values, c)
train_sorted['target_2h_norm'] = target_scaler.transform(train_sorted['target_2h'].values, 'target_2h')

train_sorted = train_sorted.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

all_X = []
all_y = []
all_rids = []
all_oids = []
all_ts = []

norm_cols = [f'{c}_norm' for c in status_cols]
feature_matrix = train_sorted[norm_cols + time_features + ['target_2h_norm']].values
targets = train_sorted['target_2h_norm'].values

route_groups = train_sorted.groupby('route_id', sort=False)

for rid, group_indices in route_groups.groups.items():
    indices = group_indices.values
    n_rows = len(indices)
    
    if n_rows <= WINDOW_SIZE + FORECAST_HORIZON:
        continue
    
    route_features = feature_matrix[indices]
    route_target = targets[indices]
    route_ts = train_sorted.loc[indices, 'timestamp'].values
    route_oid = train_sorted.loc[indices[0], 'office_from_id']
    
    for i in range(WINDOW_SIZE, n_rows - FORECAST_HORIZON + 1):
        all_X.append(route_features[i - WINDOW_SIZE:i])
        all_y.append(route_target[i:i + FORECAST_HORIZON])
        all_rids.append(rid)
        all_oids.append(route_oid)
        all_ts.append(route_ts[i])

all_X = np.array(all_X, dtype=np.float32)
all_y = np.array(all_y, dtype=np.float32)
all_rids = np.array(all_rids, dtype=np.int16)
all_oids = np.array(all_oids, dtype=np.int16)
all_ts = np.array(all_ts)

print(f"Sequences: {all_X.shape}, Targets: {all_y.shape}")

Building sequences efficiently...
Sequences: (1328350, 48, 14), Targets: (1328350, 10)


# Датасеты и даталоэдеры

In [13]:
last_10_timestamps = sorted(train['timestamp'].unique())[-10:]
last_10_timestamps_np = pd.to_datetime(last_10_timestamps).to_numpy()
print(f"\nValidation timestamps (last 10): {last_10_timestamps[0]} — {last_10_timestamps[-1]}")

val_mask = np.isin(all_ts, last_10_timestamps_np)
train_mask = ~val_mask

X_tr, y_tr = all_X[train_mask], all_y[train_mask]
r_tr, o_tr = all_rids[train_mask], all_oids[train_mask]
X_va, y_va = all_X[val_mask], all_y[val_mask]
r_va, o_va = all_rids[val_mask], all_oids[val_mask]

print(f"Train: {X_tr.shape[0]:,}, Val: {X_va.shape[0]:,}")


Validation timestamps (last 10): 2025-05-30 06:00:00 — 2025-05-30 10:30:00
Train: 1,328,040, Val: 310


In [14]:
class TSDataset(Dataset):
    def __init__(self, X, y, r, o):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.r = torch.LongTensor(r)
        self.o = torch.LongTensor(o)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.r[i], self.o[i], self.y[i]

train_loader = DataLoader(TSDataset(X_tr, y_tr, r_tr, o_tr),
                          batch_size=BS, shuffle=True, pin_memory=True)
val_loader = DataLoader(TSDataset(X_va, y_va, r_va, o_va),
                        batch_size=BS * 2, shuffle=False, pin_memory=True)

# Модель

In [15]:
class DeliveryNet(nn.Module):
    def __init__(self, nf=16, nr=1000, no=54,
                 re=32, oe=16, ch=64, lh=128, ll=2, dr=0.2, forecast_horizon=10):
        super().__init__()
        self.forecast_horizon = forecast_horizon
        
        self.r_emb = nn.Embedding(nr, re)
        self.o_emb = nn.Embedding(no, oe)

        self.cnn = nn.Sequential(
            nn.Conv1d(nf, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),
            nn.Conv1d(ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),
        )
        self.lstm = nn.LSTM(ch, lh, ll, batch_first=True,
                            dropout=dr if ll > 1 else 0)
        self.ln = nn.LayerNorm(lh)

        self.attn = nn.Sequential(
            nn.Linear(lh, lh // 2), nn.Tanh(), nn.Linear(lh // 2, 1))

        head_in = lh * 2 + re + oe
        
        self.head = nn.Sequential(
            nn.Linear(head_in, 128), nn.GELU(), nn.Dropout(dr),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dr),
            nn.Linear(64, forecast_horizon)
        )

    def forward(self, x, rid, oid):
        h = self.cnn(x.permute(0, 2, 1)).permute(0, 2, 1)
        h, _ = self.lstm(h)
        h = self.ln(h)

        w = torch.softmax(self.attn(h), dim=1)
        ctx = (h * w).sum(dim=1)

        last = h[:, -1, :]

        r = self.r_emb(rid)
        o = self.o_emb(oid)
        
        return self.head(torch.cat([ctx, last, r, o], dim=1))

In [16]:
model = DeliveryNet(nf=num_seq_features).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 336,171


In [ ]:
model.load_state_dict(torch.load('model_weights_noe_all_fourth.pth'))

# Функции ошибки

In [ ]:
import torch.nn.functional as F
class SparseWapeLoss(nn.Module):
    def __init__(self, bias_w: float = 0.5):
        super().__init__()
        self.bias_w = bias_w

    def forward(self, pred, true):
        delta  = torch.quantile(true.abs(), 0.99).detach().clamp(min=1.0)
        huber  = F.huber_loss(pred, true, delta=delta.item(), reduction='none')

        weights = 1.0 / (torch.sqrt(true.abs() + 1.0))
        loss    = (huber * weights).mean()

        s_true = true.sum()
        bias   = ((pred.sum() / (s_true.abs() + 1e-6)) - 1.0).abs()
        return loss + self.bias_w * bias

criterion = SparseWapeLoss(0.4)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-5)

In [17]:
class WeightedWapeBiasLoss(nn.Module):
    def __init__(self, bias_w=0.3):
        super().__init__()
        self.bias_w = bias_w
    
    def forward(self, pred, true):
        weights = torch.sqrt(torch.abs(true) + 1.0)
        weighted_mae = ((pred - true).abs() * weights).mean()
        bias = (pred.mean() - true.mean()).abs()
        return weighted_mae + self.bias_w * bias

criterion = WeightedWapeBiasLoss(0.3)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-5)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer,
#     mode='min',
#     factor=0.5,          
#     patience=3,          
#     min_lr=1e-6,
#     verbose=True
# )

In [18]:
best_metric, best_epoch, best_state = float('inf'), 0, None
checkpoint_path = 'best_model.pth'

# Обучение

### Дефолтная тренировка

In [19]:
print("\nTraining...")
for ep in range(EPOCHS):
    model.train()
    losses = []
    for xb, rb, ob, yb in train_loader:
        xb, rb, ob, yb = [t.to(device) for t in [xb, rb, ob, yb]]
        
        if torch.isnan(xb).any() or torch.isinf(xb).any():
            print("NaN/Inf in input!")
            print(f"xb stats: min={xb.min()}, max={xb.max()}, mean={xb.mean()}")
            break
            
        optimizer.zero_grad()
        loss = criterion(model(xb, rb, ob), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    model.eval()
    vp, vt = [], []
    with torch.no_grad(), autocast():
        for xb, rb, ob, yb in val_loader:
            vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
            vt.append(yb.numpy())
            
    vp = np.concatenate(vp)
    vt = np.concatenate(vt)

    vp_d = np.clip(target_scaler.inverse_transform(vp.flatten(), 'target_2h'), 0, None).reshape(vp.shape)
    vt_d = target_scaler.inverse_transform(vt.flatten(), 'target_2h').reshape(vt.shape)
    
    m = wape_plus_rbias(vt_d.flatten(), vp_d.flatten())
    
#     scheduler.step(m)

    if m < best_metric:
        best_metric, best_epoch = m, ep
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        torch.save({
            'epoch': ep,
            'model_state': best_state,
            'metric': best_metric,
            'optimizer_state': optimizer.state_dict(),
        }, f'best_model_ep{ep}.pth')
        print(f"  → Saved checkpoint: {checkpoint_path}")


    w = np.abs(vp_d - vt_d).sum() / vt_d.sum()
    b = np.abs(vp_d.sum() / vt_d.sum() - 1)
    print(f"Ep {ep:3d} | L {np.mean(losses):.4f} | "
          f"M {m:.4f} (W:{w:.4f} B:{b:.4f})"
          f"{' *' if m <= best_metric else ''}")

    if ep - best_epoch >= PATIENCE:
        print(f"Early stop at {ep}")
        break


Training...
  → Saved checkpoint: best_model.pth
Ep   0 | L 0.5448 | M 0.4865 (W:0.4360 B:0.0506) *
  → Saved checkpoint: best_model.pth
Ep   1 | L 0.5088 | M 0.4390 (W:0.4379 B:0.0011) *
Ep   2 | L 0.4996 | M 0.4601 (W:0.4310 B:0.0291)
  → Saved checkpoint: best_model.pth
Ep   3 | L 0.4933 | M 0.4380 (W:0.4352 B:0.0028) *
Ep   4 | L 0.4872 | M 0.4382 (W:0.4354 B:0.0028)
Ep   5 | L 0.4803 | M 0.4441 (W:0.4180 B:0.0261)
  → Saved checkpoint: best_model.pth
Ep   6 | L 0.4717 | M 0.4332 (W:0.4207 B:0.0125) *
  → Saved checkpoint: best_model.pth
Ep   7 | L 0.4639 | M 0.4325 (W:0.4129 B:0.0196) *
  → Saved checkpoint: best_model.pth
Ep   8 | L 0.4581 | M 0.4247 (W:0.4157 B:0.0090) *
  → Saved checkpoint: best_model.pth
Ep   9 | L 0.4547 | M 0.4209 (W:0.4208 B:0.0001) *
Ep  10 | L 0.4669 | M 0.4504 (W:0.4106 B:0.0399)
Ep  11 | L 0.4588 | M 0.4285 (W:0.4069 B:0.0216)
Ep  12 | L 0.4499 | M 0.4583 (W:0.4052 B:0.0531)
Ep  13 | L 0.4411 | M 0.4341 (W:0.3980 B:0.0361)
Ep  14 | L 0.4332 | M 0.4435

KeyboardInterrupt: 

In [20]:
print(f"\nBest: ep {best_epoch}, metric {best_metric:.6f}")
model.load_state_dict(best_state)
model = model.to(device)


Best: ep 54, metric 0.348526


In [ ]:
checkpoint = torch.load('best_model_ep20.pth')
model.load_state_dict(checkpoint['model_state'])

In [21]:
torch.save(model.state_dict(), 'model_weights_sparse_all_fifth.pth')

# Предсказание тестовых данных

### Дефолтное

In [ ]:
len(available_routes)

In [23]:
test = test[test.route_id.isin(train['route_id'].unique())]

In [24]:
len(test.route_id.unique())

310

In [ ]:
len(test.route_id.unique())

In [ ]:
test = train_sorted.groupby('route_id').tail(10)

In [ ]:
test = test.reset_index().rename(columns={'index':'id'})

In [ ]:
train_sorted = train_sorted[train_sorted.groupby('route_id', sort=False).cumcount().between(4341-WINDOW_SIZE-10, 4341-11)]

In [25]:
route_histories = {}

for rid in train_sorted['route_id'].unique():
    rd = train_sorted[train_sorted['route_id'] == rid]
    norm_parts = [scaler.transform(rd[c].values, c) for c in status_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    target_arr = rd['target_2h_norm'].values.reshape(-1, 1)
    full = np.column_stack([norm_arr, time_arr, target_arr]).astype(np.float32)
    route_histories[rid] = list(full[-WINDOW_SIZE:])

In [26]:
test_by_route = test.groupby('route_id', sort=False)

id_to_pred = {}

model.eval()

print("\nInference on test set...")
for rid, group_df in test_by_route:    
    oid = int(group_df.iloc[0]['office_from_id'])
    batch_ids = group_df['id'].values
    
    hist = np.array(route_histories[rid], dtype=np.float32)  # [WINDOW_SIZE, num_features]
    
    with torch.no_grad():
        pred_norm = model(
            torch.FloatTensor(hist).unsqueeze(0).to(device),  # [1, WINDOW_SIZE, num_features]
            torch.LongTensor([rid]).to(device),
            torch.LongTensor([oid]).to(device)
        ).cpu().numpy()  # [1, FORECAST_HORIZON]
    
    pred_norm = pred_norm[0]  # [FORECAST_HORIZON]
    
    # Денормализуем
    preds = np.clip(target_scaler.inverse_transform(pred_norm, 'target_2h'), 0, None)
    
    # Соответствуем ID из тестового набора
    for j, original_id in enumerate(batch_ids):
        id_to_pred[original_id] = preds[j]


Inference on test set...


In [27]:
submission = pd.DataFrame({
    'id': list(id_to_pred.keys()),
    'y_pred': list(id_to_pred.values())
})
submission = submission.sort_values('id').reset_index(drop=True)

In [28]:
submission.to_csv('submission_nn_sparse_310_end.csv', index=False)
print(f"\nDone! {submission.shape}")
print(f"Sum: {submission['y_pred'].sum():.0f}")
print(submission.head(10))


Done! (3100, 2)
Sum: 218729
   id    y_pred
0  40  5.885816
1  41  4.503114
2  42  4.816221
3  43  4.511239
4  44  4.528075
5  45  5.153872
6  46  5.842619
7  47  6.560798
8  48  7.351366
9  49  8.128387


# Диагностика проблемы

### Дефолтная

In [ ]:
model.eval()
vp, vt = [], []
with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')

errors = np.abs(vp_d - vt_d)
rel_errors = errors / (vt_d + 1e-5)

print("Error stats:")
print(f"  MAE: {errors.mean():.4f}")
print(f"  RMSE: {np.sqrt((errors**2).mean()):.4f}")
print(f"  Max error: {errors.max():.4f}")
print(f"  Median error: {np.median(errors):.4f}")
print(f"  90th percentile: {np.percentile(errors, 90):.4f}")
print(f"  % with error > 50: {(errors > 50).mean() * 100:.1f}%")

In [ ]:
val_df = pd.DataFrame({
    'route_id': r_va,
    'true': vt_d,
    'pred': vp_d
})
route_errors = val_df.groupby('route_id').apply(
    lambda g: np.abs(g['pred'] - g['true']).mean()
).sort_values(ascending=False)
print("\nWorst 10 routes:")
print(route_errors.head(10))

In [ ]:
train.groupby('route_id', sort=False)[['status_1', 'status_3']].mean().query('status_1 < 0.5 and status_3 < 0.5').index.tolist()

In [ ]:
val_df['error'] = np.abs(val_df['pred'] - val_df['true'])
val_df['step'] = val_df.groupby('route_id').cumcount()

error_by_step = val_df.groupby('step')['error'].agg(['mean', 'median', 'std'])
print("\nError by prediction step:")
print(error_by_step)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(error_by_step.index, error_by_step['mean'], marker='o', label='Mean')
plt.plot(error_by_step.index, error_by_step['median'], marker='s', label='Median')
plt.xlabel('Step (0=first, 9=last)')
plt.ylabel('Absolute Error')
plt.title('Error vs Prediction Step')
plt.legend()
plt.grid()
plt.show()